In [12]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector

In [13]:
load_dotenv(override=True)

True

In [14]:
# temperature=0 ensures deterministic entity and relationship extraction
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [15]:
# PyPDFLoader yields one Document per page
loader = PyPDFLoader("data/elon_musk.pdf")
pages = loader.load()

for i, p in enumerate(pages):
    print(f"Page {i + 1}: {len(p.page_content)} chars")

incorrect startxref pointer(1)
parsing for Object Streams


Page 1: 2343 chars
Page 2: 1192 chars


In [16]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")

14 chunks created


In [17]:
print("Neo4j credentials loaded:", bool(os.environ.get("NEO4J_PASSWORD")))

Neo4j credentials loaded: True


In [18]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database=os.environ["NEO4J_DATABASE"],
)

In [19]:
graph_transformer = LLMGraphTransformer(llm=llm)

In [20]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

14 graph documents extracted
Nodes: ['Elon Musk', 'June 28, 1971', 'Pretoria', 'South Africa', 'American', 'Entrepreneur', 'Engineer', "World'S Wealthiest Person"]
Rels:  [('Elon Musk', 'BORN_IN', 'Pretoria'), ('Elon Musk', 'BORN_ON', 'June 28, 1971'), ('Elon Musk', 'HAS_NATIONALITY', 'American'), ('Elon Musk', 'HAS_OCCUPATION', 'Entrepreneur'), ('Elon Musk', 'HAS_OCCUPATION', 'Engineer'), ('Elon Musk', 'RECOGNISED_AS', "World'S Wealthiest Person"), ('Pretoria', 'LOCATED_IN', 'South Africa')]


In [22]:
# include_source=True links each entity node back to its source Document node,
# which is required for Neo4jVector.from_existing_graph in the next cell
graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)
print("Graph stored in Neo4J")

Graph stored in Neo4J


In [23]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database=os.environ["NEO4J_DATABASE"],
    index_name="elon_musk_chunks",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")

Vector index created


In [24]:
# verify what landed in Neo4J
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
for r in node_counts:
    print(" ", r)
print("Relationships:")
for r in rel_counts:
    print(" ", r)

Nodes:
  {'label': ['__Entity__', 'Person'], 'count': 23}
  {'label': ['Document'], 'count': 14}
  {'label': ['__Entity__', 'Location'], 'count': 12}
  {'label': ['__Entity__', 'Product'], 'count': 10}
  {'label': ['__Entity__', 'Date'], 'count': 7}
  {'label': ['__Entity__', 'Organization'], 'count': 7}
  {'label': ['__Entity__', 'Occupation'], 'count': 5}
  {'label': ['__Entity__', 'Company'], 'count': 4}
  {'label': ['__Entity__', 'Nationality'], 'count': 3}
  {'label': ['__Entity__', 'Concept'], 'count': 3}
  {'label': ['__Entity__', 'Country'], 'count': 2}
  {'label': ['__Entity__', 'Organization', 'Company'], 'count': 2}
  {'label': ['__Entity__', 'Service'], 'count': 2}
  {'label': ['__Entity__', 'Place'], 'count': 1}
  {'label': ['__Entity__', 'Title'], 'count': 1}
  {'label': ['__Entity__', 'Amount'], 'count': 1}
  {'label': ['__Entity__', 'Year'], 'count': 1}
  {'label': ['__Entity__', 'Date', 'Year'], 'count': 1}
  {'label': ['__Entity__', 'Technology'], 'count': 1}
  {'labe